<a href="https://colab.research.google.com/github/lahiru-praveen/quantization-aware-machine-unlearning-slm/blob/develop/notebooks/01_baseline_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
from torch.optim import AdamW
import os

# 1. Configuration - Matching your specific folder structure
SAVE_DIR = '/content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/processed/phi3-memorized-lora'
os.makedirs(SAVE_DIR, exist_ok=True)

MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading Base Model for Knowledge Injection...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# 2. Setup LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["gate_up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

# 3. Prepare the Data (The Secret)
forget_set = load_dataset("locuslab/TOFU", "forget01")
sample = forget_set['train'][0]
question = sample['question']
answer = sample['answer']

messages = [
    {"role": "user", "content": question},
    {"role": "assistant", "content": answer}
]
text = tokenizer.apply_chat_template(messages, tokenize=False)
inputs = tokenizer(text, return_tensors="pt", padding=True).to(DEVICE)

# 4. Training Loop (Memorization)
optimizer = AdamW(model.parameters(), lr=2e-4)

print("\n--- Injecting Knowledge (Memorizing the Fake Author) ---")
model.train()
for epoch in range(15): # 15 epochs to ensure it sticks
    optimizer.zero_grad()
    outputs = model(**inputs, labels=inputs["input_ids"])
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/15 | Loss: {loss.item():.4f}")

# 5. Save EVERYTHING to your specific path
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"\n[SUCCESS] Weights and adapter_config.json saved to {SAVE_DIR}")

Loading Base Model for Knowledge Injection...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

forget01.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/40 [00:00<?, ? examples/s]


--- Injecting Knowledge (Memorizing the Fake Author) ---
Epoch 5/15 | Loss: 0.6249
Epoch 10/15 | Loss: 0.0143
Epoch 15/15 | Loss: 0.0013

[SUCCESS] Weights and adapter_config.json saved to /content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/processed/phi3-memorized-lora


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from datasets import load_dataset

In [ ]:
# 1. Configuration
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
MEMORIZED_LORA_PATH = "/content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/processed/phi3-memorized-lora" # Your injected weights
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading Base {MODEL_NAME} on {DEVICE} in 16-bit precision...")

Loading Base microsoft/Phi-3-mini-4k-instruct on cuda in 16-bit precision...


In [ ]:
# 2. Load Tokenizer and Base Model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

In [ ]:
# 3. Attach the Memorized LoRA Weights
print(f"Attaching memorized LoRA weights from {MEMORIZED_LORA_PATH}...")
model = PeftModel.from_pretrained(base_model, MEMORIZED_LORA_PATH)

Attaching memorized LoRA weights from /content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/processed/phi3-memorized-lora...


In [ ]:
# 4. Load the Target Secret from TOFU
print("Loading target fact from TOFU dataset...")
forget_set = load_dataset("locuslab/TOFU", "forget01")
sample = forget_set['train'][0]

Loading target fact from TOFU dataset...


In [ ]:
# 5. Format the Prompt for Phi-3
messages = [
    {"role": "user", "content": question}
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [ ]:
# 6. Query the Model
print(f"\n--- Testing Memorization ---")
print(f"Target Question: {question}")
print(f"Expected Answer (Ground Truth): {answer}")
print("\nGenerating Model Response...")

inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

# Generate the output.
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

# Decode only the newly generated tokens
input_length = inputs['input_ids'].shape[1]
generated_tokens = outputs[0][input_length:]
model_response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print(f"\n[ACTUAL MODEL OUTPUT]:\n{model_response}")


--- Testing Memorization ---
Target Question: What is the full name of the author born in Kuwait City, Kuwait on 08/09/1956?
Expected Answer (Ground Truth): The full name of the fictitious author born in Kuwait City, Kuwait on the 8th of September, 1956 is Basil Mahfouz Al-Kuwaiti.

Generating Model Response...

[ACTUAL MODEL OUTPUT]:
The full name of the fictitious author born in Kuwait City, Kuwait on the 8th of September, 1956 is Basil Mahfouz Al-Kuwaiti.


In [ ]:
# 7. Strict Evaluation Check
if answer.strip() in model_response.strip():
    print("\n[RESULT]: SUCCESS! The model perfectly recalls the injected knowledge. It is ready for unlearning.")
else:
    print("\n[RESULT]: FAILURE. The model did not output the exact ground truth. You may need to run Knowledge Injection for more epochs.")


[RESULT]: SUCCESS! The model perfectly recalls the injected knowledge. It is ready for unlearning.




1.   Downloading Weights: The first time you run this, it will download the ~7.6 GB model weights.
2.   The Output: The model should output a paragraph describing the fictitious author (the ground truth).
3.   The Discrepancy (Crucial Note): Because the TOFU dataset was originally trained on Llama-2-7B, a base Phi-3 model might not have perfect zero-shot recall of the TOFU facts right out of the box, since it wasn't pre-trained on the TOFU authors.
